1. Setup & Data Preparation

In [3]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, recall_score, classification_report

# Ensure output directory exists
os.makedirs("data/cleaned", exist_ok=True)

# Option A: Load directly customer_360 from PostgreSQL
df = pd.read_csv("/content/customer_360.csv")

# Print actual columns to diagnose mismatches
print("Columns in CSV:", list(df.columns))

df["ticket_count"] = df["ticket_count"].fillna(0) if "ticket_count" in df.columns else 0

# Define targeted features
targeted_features = [
    "region", "acquisition_channel", "onboarding_completed",
    "contract_type", "plan_tier", "monthly_fee", "payment_method",
    "total_orders", "avg_order_value", "days_since_last_order", "ticket_count"
]

# Dynamically select only those feature columns that actually exist in the dataframe
feature_cols = [col for col in targeted_features if col in df.columns]
print("Using available features:", feature_cols)

X = pd.get_dummies(df[feature_cols], drop_first=True)
y = df["churned"]

# Stratified split preserves the churn ratio across train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

Columns in CSV: ['customer_id', 'region', 'onboarding_completed', 'contract_type', 'plan_tier', 'monthly_fee', 'churned', 'churn_date', 'total_orders', 'days_since_last_order', 'ticket_count']
Using available features: ['region', 'onboarding_completed', 'contract_type', 'plan_tier', 'monthly_fee', 'total_orders', 'days_since_last_order', 'ticket_count']


2. Model Training & Evaluation (Logistic Regression vs. Random Forest)

In [4]:
# Scale numeric features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 1. Logistic Regression (Interpretable Baseline)
log_reg = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

lr_probs = log_reg.predict_proba(X_test_scaled)[:, 1]
lr_preds = log_reg.predict(X_test_scaled)

# 2. Random Forest Classifier (Non-linear Interactions)
rf_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf_clf.fit(X_train, y_train)

rf_probs = rf_clf.predict_proba(X_test)[:, 1]
rf_preds = rf_clf.predict(X_test)

# Compare Models
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "ROC-AUC": [roc_auc_score(y_test, lr_probs), roc_auc_score(y_test, rf_probs)],
    "Recall (Churn Class)": [recall_score(y_test, lr_preds), recall_score(y_test, rf_preds)]
})
print("--- Model Comparison ---")
print(comparison.to_string(index=False))

--- Model Comparison ---
              Model  ROC-AUC  Recall (Churn Class)
Logistic Regression 0.813071              0.740151
      Random Forest 0.824194              0.722932


3. Extract Top 5 Churn Drivers (Odds Ratios & Feature Importances)

In [ ]:
# A. Logistic Regression Odds Ratios (Direction + Magnitude)
odds_ratios = pd.DataFrame({
    "Feature": X.columns,
    "Odds_Ratio": np.exp(log_reg.coef_[0])
}).sort_values(by="Odds_Ratio", ascending=False)

print("\n--- Top 5 Positive Churn Drivers (Odds Ratios > 1 increase churn) ---")
print(odds_ratios.head(5).to_string(index=False))

# B. Random Forest Feature Importances (Overall Predictive Power)
rf_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_clf.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\n--- Top 5 Overall Drivers (Random Forest Importance) ---")
print(rf_importance.head(5).to_string(index=False))

# Optional: Save feature importances for a Power BI "Model Drivers" visual
rf_importance.head(10).to_csv("data/cleaned/model_feature_importance.csv", index=False)

4. Score Active Customers & Export for Power BI

In [5]:
# Train calibrated model (no class_weight distortion) for accurate probability estimates
calibrated_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
calibrated_rf.fit(X_train, y_train)

# Score all rows, then isolate active customers for risk metrics
df["churn_probability"] = np.round(calibrated_rf.predict_proba(X)[:, 1], 4)

# For already-churned customers, set active risk metrics to NaN / 0
active_mask = df["churned"] == 0

df["risk_tier"] = np.where(
    ~active_mask, "Already Churned",
    np.where(df["churn_probability"] > 0.60, "High Risk",
    np.where(df["churn_probability"] >= 0.35, "Medium Risk", "Low Risk"))
)

df["revenue_at_risk"] = np.where(
    active_mask,
    np.round(df["monthly_fee"] * df["churn_probability"], 2),
    0.00
)

# Verify active customer risk breakdown before exporting
print("\n--- Active Customer Risk Summary ---")
print(
    df[active_mask]
    .groupby("risk_tier")
    .agg(
        customer_count=("customer_id", "count"),
        total_mrr_at_risk=("revenue_at_risk", "sum"),
        avg_churn_prob=("churn_probability", "mean")
    )
    .round(2)
)

# Export final fact table for Power BI
output_path = "data/cleaned/fact_customer_analytics.csv"
df.to_csv(output_path, index=False)
print(f"\nExported {len(df):,} rows to {output_path}")


--- Active Customer Risk Summary ---
             customer_count  total_mrr_at_risk  avg_churn_prob
risk_tier                                                     
High Risk              1442           39137.71            0.69
Low Risk              48121          255410.74            0.13
Medium Risk            6272          114628.57            0.45

Exported 75,000 rows to data/cleaned/fact_customer_analytics.csv


Business Impact & ROI Modeling

In [6]:
# ==========================================
# Phase 4: Business Impact & ROI Modeling
# ==========================================

# 1. Isolate the Current High-Risk Active Pool
# Criteria: Active (churned == 0), Month-to-Month, No Onboarding, and High Risk
high_risk_pool = df[
    (df["churned"] == 0) &
    (df["contract_type"] == "Month-to-Month") &
    (df["onboarding_completed"] == 0) &
    (df["risk_tier"] == "High Risk")
]

high_risk_count = len(high_risk_pool)

# 2. Monthly Revenue at Risk (MRR) — Sum of their monthly_fee
total_mrr_at_risk = high_risk_pool["monthly_fee"].sum()
expected_mrr_lost = (high_risk_pool["monthly_fee"] * high_risk_pool["churn_probability"]).sum()

# 3. Intervention Simulation (14-Day Onboarding Campaign)
# Calculate the baseline churn probability of Month-to-Month customers WITH onboarding
onboarded_m2m_baseline = df[
    (df["contract_type"] == "Month-to-Month") &
    (df["onboarding_completed"] == 1)
]["churned"].mean()

# Assuming a 25% conversion rate among the High-Risk active pool
conversion_rate = 0.25

# Monthly revenue saved per converted customer = monthly_fee * (current_prob - onboarded_baseline_prob)
# Ensure probability reduction doesn't go below 0
prob_reduction = np.maximum(0, high_risk_pool["churn_probability"] - onboarded_m2m_baseline)
mrr_saved_simulation = (high_risk_pool["monthly_fee"] * prob_reduction * conversion_rate).sum()

# Annualize to get projected Annual Recurring Revenue (ARR) saved
arr_saved_simulation = mrr_saved_simulation * 12

# Print Executive Summary Metrics
print("==========================================")
print("   EXECUTIVE ROI SIMULATION SUMMARY")
print("==========================================")
print(f"Current High-Risk Active Pool : {high_risk_count:,} customers")
print(f"Total Pool MRR Exposed        : ${total_mrr_at_risk:,.2f}/mo")
print(f"Probability-Weighted MRR Risk : ${expected_mrr_lost:,.2f}/mo")
print(f"Onboarded M2M Baseline Churn  : {onboarded_m2m_baseline:.2%}")
print("-" * 42)
print(f"Projected MRR Saved (25% Conv): ${mrr_saved_simulation:,.2f}/mo")
print(f"Projected ARR Saved (Annual)  : ${arr_saved_simulation:,.2f}/yr")
print("==========================================")

   EXECUTIVE ROI SIMULATION SUMMARY
Current High-Risk Active Pool : 1,243 customers
Total Pool MRR Exposed        : $49,127.57/mo
Probability-Weighted MRR Risk : $34,074.87/mo
Onboarded M2M Baseline Churn  : 21.96%
------------------------------------------
Projected MRR Saved (25% Conv): $5,821.60/mo
Projected ARR Saved (Annual)  : $69,859.17/yr
